<a href="https://colab.research.google.com/github/TarunSinghal960/RAG-AI-for-Indian-Penal-Code/blob/main/RAG_AI_for_Indian_Penal_Code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install langchain langchain-community langchain-core langchain-text-splitters faiss-cpu sentence-transformers pypdf langchain-google-genai google-generativeai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 68.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 329.1/329.1 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.5 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.43.0, but you have google-auth 2.48.0 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is inco

In [ ]:
import os
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

DATA_PATH = "/content/drive/My Drive/RAG AI IPC/ipc_docs"
DB_PATH = "/content/drive/My Drive/RAG AI IPC/vector_db"

documents = []

for file in os.listdir(DATA_PATH):
    file_path = os.path.join(DATA_PATH, file)

    if file.endswith(".pdf"):
        loader = PyPDFLoader(file_path)
    elif file.endswith(".txt"):
        loader = TextLoader(file_path)
    else:
        continue

    documents.extend(loader.load())

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150
)

chunks = text_splitter.split_documents(documents)

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vector_db = FAISS.from_documents(chunks, embeddings)
vector_db.save_local(DB_PATH)

print("✅ Vector DB created successfully")

/tmp/ipython-input-3577440920.py:31: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Vector DB created successfully


In [ ]:
import os
import google.generativeai as genai
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# API Key
API_KEY = os.getenv("GOOGLE_API_KEY")
if not API_KEY:
    raise ValueError("GOOGLE_API_KEY not found in environment variables")

genai.configure(api_key=API_KEY)

DB_PATH = "/content/drive/My Drive/RAG AI IPC/vector_db"

# Load embeddings
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Load FAISS DB
vector_db = FAISS.load_local(
    DB_PATH,
    embeddings,
    allow_dangerous_deserialization=True
)

# FREE TIER MODEL (GUARANTEED)
model = genai.GenerativeModel("models/gemini-2.5-flash")

print("Google FREE-tier IPC RAG system ready")

while True:
    query = input("\nAsk about IPC (type 'exit'): ")
    if query.lower() == "exit":
        break

    # Retrieve IPC chunks
    docs = vector_db.similarity_search(query, k=4)
    context = "\n\n".join(doc.page_content for doc in docs)

    prompt = f"""
You are a legal assistant specialized in the Indian Penal Code (IPC).
Answer ONLY using the context below.
If the answer is not found, say:
"Information not found in the IPC documents."

Context:
{context}

Question:
{query}

Answer:
"""

    response = model.generate_content(prompt)
    print("\n📌 Answer:\n", response.text)

    print("\n📚 Sources:")
    for doc in docs:
        print("-", doc.metadata.get("source"))

✅ Google FREE-tier IPC RAG system ready

Ask about IPC (type 'exit'): punishment for domestic violence

📌 Answer:
 Whoever, being the husband or the relative of the husband of a woman, subjects such woman to cruelty shall be punished with imprisonment for a term which may extend to three years and shall also be liable to fine.

📚 Sources:
- /content/drive/My Drive/RAG AI IPC/ipc_docs/repealedfileopen.pdf
- /content/drive/My Drive/RAG AI IPC/ipc_docs/repealedfileopen.pdf
- /content/drive/My Drive/RAG AI IPC/ipc_docs/repealedfileopen.pdf
- /content/drive/My Drive/RAG AI IPC/ipc_docs/repealedfileopen.pdf

Ask about IPC (type 'exit'): punishment for false case

📌 Answer:
 Based on the IPC documents provided:

*   **For dishonestly making a false claim in Court (Section 209):** The punishment is imprisonment of either description for a term which may extend to two years, and the person shall also be liable to fine.
*   **For intentionally giving false evidence or fabricating false evidence 